Pointer net results

In [ ]:
# ============================================================
# EVALUATE MODEL ACROSS 100 DIFFERENT SEEDS
# ============================================================

import numpy as np
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
from utils import mask_fn
from models.transformer_policy import MaskableTransformerPolicy

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from envs.stack_gym import StackEnv

# ============================================================
# CONFIG (must match training config exactly)
# ============================================================

config_dict = {
            "vessel_shape": (4, 4, 4),
            "yard_shape": (4, 4, 4),
            "num_containers": 64,
            "group_num": 4,
            "group_placement": "random",
            "observation_type": "stack_features",
            "reward_norm": False
            }

# ============================================================
# LOAD MODEL
# ============================================================

model = MaskablePPO.load("models/ppo_stack_medium_clip_parallelenv_transsmallnet_lr1e4_best",
                         custom_objects={"policy_class": MaskableTransformerPolicy}
                         )

print("=" * 70)
print("EVALUATING MODEL ACROSS 500 SEEDS")
print("=" * 70)

# ============================================================
# RUN 500 EPISODES WITH DIFFERENT SEEDS
# ============================================================

steps = config_dict["num_containers"]
num_seeds = 500
all_rewards_per_Step = np.zeros((num_seeds, steps))
all_rewards = []

for seed in range(num_seeds):
    
    # Create eval env
    eval_env = StackEnv(config=config_dict, render_mode=None)
    eval_env = ActionMasker(eval_env, mask_fn)
    
    # Reset with specific seed
    obs, info = eval_env.reset(seed=seed)
    terminated = False
    truncated = False
    total_reward = 0.0
    step_count = 0
    
    # Run episode
    while not (terminated or truncated) and step_count < 120:
        action, _ = model.predict(
            obs,
            deterministic=True,
            action_masks=get_action_masks(eval_env)
        )
        
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        step_count += 1
        all_rewards_per_Step[seed, step_count - 1] = reward
        
    
    all_rewards.append(total_reward)
    eval_env.close()
    
    # Progress update every 50 seeds
    if (seed + 1) % 50 == 0:
        print(f"Completed {seed + 1}/{num_seeds} seeds...")

# ============================================================
# COMPUTE STATISTICS
# ============================================================

all_rewards = np.array(all_rewards)
mean_reward = np.mean(all_rewards)
std_reward = np.std(all_rewards)
min_reward = np.min(all_rewards)
max_reward = np.max(all_rewards)

percent_of_seeds_with_max_reward = np.sum(all_rewards > 416) / num_seeds * 100

print(f"Seed with highest reward: {np.argmax(all_rewards)}")
print(f"Seed with lowest reward: {np.argmin(all_rewards)}")


print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)
print(f"Mean Reward: {mean_reward:.4f}")
print(f"Std Reward:  {std_reward:.4f}")
print(f"Min Reward:  {min_reward:.4f}")
print(f"Max Reward:  {max_reward:.4f}")
print(f"Percentage of seeds that achieved max reward: {percent_of_seeds_with_max_reward:.2f}%")
print("=" * 70)



# Density plots with 
plt.figure()

# histogram
plt.hist(all_rewards, bins=30, density=True)

# density curve
density = gaussian_kde(all_rewards)
xs = np.linspace(all_rewards.min(), all_rewards.max(), 200)
plt.plot(xs, density(xs))

plt.xlabel("Final Episode Reward")
plt.ylabel("Density")
plt.title("Distribution Plot of Final Episode Rewards for 500 Seeds")
plt.show()

In [ ]:
# ============================================================
# LOAD SAVED MODEL + INTERACTIVE VISUAL EVALUATION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
from utils import mask_fn
from models.transformer_policy import MaskableTransformerPolicy
from envs.stack_gym import StackEnv

# ============================================================
# CONFIG (must match training config exactly)
# ============================================================

config_dict = {
    "vessel_shape": (4, 4, 4),
    "yard_shape": (4, 4, 4),
    "num_containers": 64,
    "group_num": 4,
    "group_placement": "random",
    "observation_type": "stack_features",
    "reward_norm": False
}

# ============================================================
# CREATE ENV + LOAD MODEL
# ============================================================

eval_env = StackEnv(config=config_dict, render_mode="rgb_array")
eval_env = ActionMasker(eval_env, mask_fn)

model = MaskablePPO.load(
    "models/ppo_stack_medium_clip_parallelenv_transsmallnet_lr1e4_best",
    custom_objects={"policy_class": MaskableTransformerPolicy}
)

# ============================================================
# RUN EPISODE & COLLECT FRAMES
# ============================================================

obs, info = eval_env.reset(seed=20)
terminated = False
truncated = False
total_reward = 0.0
step_count = 0

frames = []  # list of (img, step, action, reward, cumulative_reward)

while not (terminated or truncated) and step_count < 120:
    action, _ = model.predict(
        obs, deterministic=True, action_masks=get_action_masks(eval_env)
    )
    obs, reward, terminated, truncated, info = eval_env.step(action)
    total_reward += reward
    step_count += 1

    img = eval_env.render()
    if img is not None:
        frames.append((img, step_count, int(action), reward, total_reward))

eval_env.close()

print(f"Episode finished: {len(frames)} frames | Total Reward: {total_reward:.4f}")

# ============================================================
# INTERACTIVE VIEWER WIDGET
# ============================================================

slider = widgets.IntSlider(
    value=0, min=0, max=len(frames) - 1, step=1,
    description="Step:", continuous_update=False,
    layout=widgets.Layout(width="60%")
)
prev_btn = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="80px"))
next_btn = widgets.Button(description="Next ▶", layout=widgets.Layout(width="80px"))
info_label = widgets.HTML()
out = widgets.Output()

def show_frame(idx):
    img, step, action, reward, cum_reward = frames[idx]
    info_label.value = (
        f"<b>Step {step}/{len(frames)}</b> &nbsp;|&nbsp; "
        f"Action: {action} &nbsp;|&nbsp; "
        f"Reward: {reward:.4f} &nbsp;|&nbsp; "
        f"Cumulative: {cum_reward:.4f}"
    )
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.imshow(img)
        ax.set_title(
            f"Step {step} | Action: {action} | "
            f"Reward: {reward:.4f} | Total: {cum_reward:.4f}"
        )
        ax.axis("off")
        fig.tight_layout()
        plt.show()

def on_slider_change(change):
    show_frame(change["new"])

def on_prev(b):
    slider.value = max(0, slider.value - 1)

def on_next(b):
    slider.value = min(len(frames) - 1, slider.value + 1)

slider.observe(on_slider_change, names="value")
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)

ui = widgets.VBox([
    info_label,
    widgets.HBox([prev_btn, slider, next_btn]),
    out
])

display(ui)
show_frame(0)

Flat RL eval

In [ ]:
# ============================================================
# EVALUATE MODEL ACROSS 100 DIFFERENT SEEDS
# ============================================================

import numpy as np
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
from utils import mask_fn


import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde


from envs.stack_gym import StackEnv

# ============================================================
# CONFIG (must match training config exactly)
# ============================================================

config_dict = {
            "vessel_shape": (4, 4, 4),
            "yard_shape": (4, 4, 4),
            "num_containers": 64,
            "group_num": 4,
            "group_placement": "random",
            "observation_type": "stack_features",
            "reward_norm": False
            }

# ============================================================
# LOAD MODEL
# ============================================================

model = MaskablePPO.load("models/ppo_stack_medium_clip_parallelenv_best")


print("=" * 70)
print("EVALUATING MODEL ACROSS 500 SEEDS")
print("=" * 70)

# ============================================================
# RUN 500 EPISODES WITH DIFFERENT SEEDS
# ============================================================

steps = config_dict["num_containers"]
num_seeds = 500
all_rewards_per_Step = np.zeros((num_seeds, steps))
all_rewards = []

for seed in range(num_seeds):
    
    # Create eval env
    eval_env = StackEnv(config=config_dict, render_mode=None)
    eval_env = ActionMasker(eval_env, mask_fn)
    
    # Reset with specific seed
    obs, info = eval_env.reset(seed=seed)
    terminated = False
    truncated = False
    total_reward = 0.0
    step_count = 0
    
    # Run episode
    while not (terminated or truncated) and step_count < 120:
        action, _ = model.predict(
            obs,
            deterministic=True,
            action_masks=get_action_masks(eval_env)
        )
        
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        step_count += 1
        all_rewards_per_Step[seed, step_count - 1] = reward
        
    
    all_rewards.append(total_reward)
    eval_env.close()
    
    # Progress update every 50 seeds
    if (seed + 1) % 50 == 0:
        print(f"Completed {seed + 1}/{num_seeds} seeds...")

# ============================================================
# COMPUTE STATISTICS
# ============================================================

all_rewards = np.array(all_rewards)
mean_reward = np.mean(all_rewards)
std_reward = np.std(all_rewards)
min_reward = np.min(all_rewards)
max_reward = np.max(all_rewards)

percent_of_seeds_with_max_reward = np.sum(all_rewards > 416) / num_seeds * 100

print(f"Seed with highest reward: {np.argmax(all_rewards)}")
print(f"Seed with lowest reward: {np.argmin(all_rewards)}")


print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)
print(f"Mean Reward: {mean_reward:.4f}")
print(f"Std Reward:  {std_reward:.4f}")
print(f"Min Reward:  {min_reward:.4f}")
print(f"Max Reward:  {max_reward:.4f}")
print(f"Percentage of seeds that achieved max reward: {percent_of_seeds_with_max_reward:.2f}%")
print("=" * 70)



# Density plots with 
plt.figure()

# histogram
plt.hist(all_rewards, bins=30, density=True)

# density curve
density = gaussian_kde(all_rewards)
xs = np.linspace(all_rewards.min(), all_rewards.max(), 200)
plt.plot(xs, density(xs))

plt.xlabel("Final Episode Reward")
plt.ylabel("Density")
plt.title("Distribution Plot of Final Episode Rewards for 500 Seeds")
plt.show()

In [ ]:
# ============================================================
# LOAD SAVED MODEL + INTERACTIVE VISUAL EVALUATION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
from utils import mask_fn
from models.transformer_policy import MaskableTransformerPolicy
from envs.stack_gym import StackEnv

# ============================================================
# CONFIG (must match training config exactly)
# ============================================================

config_dict = {
    "vessel_shape": (4, 4, 4),
    "yard_shape": (4, 4, 4),
    "num_containers": 64,
    "group_num": 4,
    "group_placement": "random",
    "observation_type": "stack_features",
    "reward_norm": False
}

# ============================================================
# CREATE ENV + LOAD MODEL
# ============================================================

eval_env = StackEnv(config=config_dict, render_mode="rgb_array")
eval_env = ActionMasker(eval_env, mask_fn)

model = MaskablePPO.load("models/ppo_stack_medium_clip_parallelenv_best")

# ============================================================
# RUN EPISODE & COLLECT FRAMES
# ============================================================

obs, info = eval_env.reset(seed=0)
terminated = False
truncated = False
total_reward = 0.0
step_count = 0

frames = []  # list of (img, step, action, reward, cumulative_reward)

while not (terminated or truncated) and step_count < 120:
    action, _ = model.predict(
        obs, deterministic=True, action_masks=get_action_masks(eval_env)
    )
    obs, reward, terminated, truncated, info = eval_env.step(action)
    total_reward += reward
    step_count += 1

    img = eval_env.render()
    if img is not None:
        frames.append((img, step_count, int(action), reward, total_reward))

eval_env.close()

print(f"Episode finished: {len(frames)} frames | Total Reward: {total_reward:.4f}")

# ============================================================
# INTERACTIVE VIEWER WIDGET
# ============================================================

slider = widgets.IntSlider(
    value=0, min=0, max=len(frames) - 1, step=1,
    description="Step:", continuous_update=False,
    layout=widgets.Layout(width="60%")
)
prev_btn = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="80px"))
next_btn = widgets.Button(description="Next ▶", layout=widgets.Layout(width="80px"))
info_label = widgets.HTML()
out = widgets.Output()

def show_frame(idx):
    img, step, action, reward, cum_reward = frames[idx]
    info_label.value = (
        f"<b>Step {step}/{len(frames)}</b> &nbsp;|&nbsp; "
        f"Action: {action} &nbsp;|&nbsp; "
        f"Reward: {reward:.4f} &nbsp;|&nbsp; "
        f"Cumulative: {cum_reward:.4f}"
    )
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.imshow(img)
        ax.set_title(
            f"Step {step} | Action: {action} | "
            f"Reward: {reward:.4f} | Total: {cum_reward:.4f}"
        )
        ax.axis("off")
        fig.tight_layout()
        plt.show()

def on_slider_change(change):
    show_frame(change["new"])

def on_prev(b):
    slider.value = max(0, slider.value - 1)

def on_next(b):
    slider.value = min(len(frames) - 1, slider.value + 1)

slider.observe(on_slider_change, names="value")
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)

ui = widgets.VBox([
    info_label,
    widgets.HBox([prev_btn, slider, next_btn]),
    out
])

display(ui)
show_frame(0)